<a href="https://colab.research.google.com/github/dee431/Student-Academic-Risk-Prediction-Best-Fit-Line/blob/main/%F0%9F%8F%A5_HHS_Unaccompanied_Children_Care_%26_Transition_Efficiency.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Cell 1: Environment Setup**

In [1]:
!pip install -q streamlit scikit-learn
!npm install -g localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 58.0 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧
added 22 packages in 2s
⠧
⠧3 packages are looking for funding
⠧  run `npm fund` for details
⠧

# **Cell 2: Data Preprocessing & Model Training**

In [4]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, root_mean_squared_error, r2_score

# 1. Load and clean the dataset
df = pd.read_csv('HHS_Unaccompanied_Alien_Children_Program.csv')
df = df.dropna()

# Convert comma-separated string numbers to float for the HHS Care column
if df['Children in HHS Care'].dtype == object:
    df['Children in HHS Care'] = df['Children in HHS Care'].str.replace(',', '').astype(float)

# 2. Define features for Care Transition Efficiency and Placement Outcome
X = df[['Children apprehended and placed in CBP custody*',
        'Children in CBP custody',
        'Children transferred out of CBP custody',
        'Children in HHS Care']]

# Target: Efficiency of transition (Discharged children)
y = df['Children discharged from HHS Care']

# 3. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Train Random Forest Regressor
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 5. Evaluate and save
predictions = model.predict(X_test)
print(f"Model R2 Score: {r2_score(y_test, predictions):.3f}")
print(f"Model Root Mean Squared Error: {root_mean_squared_error(y_test, predictions):.3f}")

joblib.dump(model, 'care_transition_model.pkl')
print("Model saved as 'care_transition_model.pkl'")

Model R2 Score: 0.872
Model Root Mean Squared Error: 42.587
Model saved as 'care_transition_model.pkl'


# **Cell 3: Creating and Running the Streamlit App**

In [5]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib
import numpy as np

# Load the trained model
model = joblib.load('care_transition_model.pkl')

st.set_page_config(page_title="Care Transition Efficiency App", layout="centered")

st.title("🏥 HHS Unaccompanied Children Care & Transition Efficiency")
st.write("""
This application predicts the number of children expected to be **discharged from HHS Care**
based on metrics of children in custody and care.
""")

st.header("Input Metrics")

# Create input form for predictions
with st.form("prediction_form"):
    apprehended = st.number_input("Children apprehended and placed in CBP custody*", min_value=0, value=100)
    in_cbp = st.number_input("Children in CBP custody", min_value=0, value=200)
    transferred = st.number_input("Children transferred out of CBP custody", min_value=0, value=180)
    in_hhs = st.number_input("Children in HHS Care", min_value=0, value=5000)

    submit_val = st.form_submit_val = st.form_submit_button("Predict Discharged Children")

if submit_val:
    # Format features matching model input
    features = np.array([[apprehended, in_cbp, transferred, in_hhs]])
    prediction = model.predict(features)[0]

    st.success(f"Predicted Number of Discharged Children: **{int(round(prediction))}**")

    # Efficiency Metric
    efficiency_ratio = (prediction / in_hhs) * 100 if in_hhs > 0 else 0
    st.metric(label="Estimated Transition Efficiency (Discharges/HHS Care)", value=f"{efficiency_ratio:.2f}%")

Writing app.py


In [ ]:
# Run Streamlit in the background with CORS and XSRF protection disabled to avoid import failures in the browser
import subprocess
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"])

# Expose the port using localtunnel
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹your url is: https://angry-crews-refuse.loca.lt
